# Task 2: Predicting Next-Day Stock Closing Price

**Objective:** Use historical stock data to predict the next day's closing price.

**Dataset:** Apple (AAPL) historical data via `yfinance` — fetched live, no file needed.

**Approach:**
- Features: Open, High, Low, Volume → Target: next day's Close
- Models: Linear Regression + Random Forest (compare both)
- Evaluation: MAE, RMSE, R²


In [ ]:
# install libraries (colab-safe)
!pip install yfinance scikit-learn matplotlib pandas --quiet

In [ ]:
# --- imports ---
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 5)

## 1. Fetch Historical Data

In [ ]:
# pull 3 years of daily AAPL data from yahoo finance
ticker = 'AAPL'
df = yf.download(ticker, start='2021-01-01', end='2024-01-01', progress=False)

print(f'downloaded {len(df)} trading days for {ticker}')
df.head()

## 2. Feature Engineering — Predicting Next Day's Close

In [ ]:
# the target is tomorrow's closing price
# we shift Close up by 1 so each row's target = next row's close
df = df.copy()
df['Target'] = df['Close'].shift(-1)

# add a few simple derived features that traders actually use
df['Price_Range'] = df['High'] - df['Low']          # daily volatility
df['Close_Open_Gap'] = df['Close'] - df['Open']     # intraday momentum
df['MA_5'] = df['Close'].rolling(5).mean()          # 5-day moving average
df['MA_20'] = df['Close'].rolling(20).mean()        # 20-day moving average

# drop the last row (no target) and any NaNs from rolling averages
df.dropna(inplace=True)

print(f'clean dataset: {df.shape[0]} rows, {df.shape[1]} columns')

## 3. Prepare Features and Split Data

In [ ]:
# features we'll use to predict tomorrow's close
feature_cols = ['Open', 'High', 'Low', 'Volume', 'Price_Range', 'Close_Open_Gap', 'MA_5', 'MA_20']

# flatten columns if yfinance returned MultiIndex
df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]

X = df[feature_cols].values
y = df['Target'].values

# keep chronological order — no shuffling for time series!
# 80% train, 20% test (last 20% = most recent data)
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
dates_test = df.index[split:]  # we'll use this for plotting

# scale features — helps linear regression a lot, doesn't hurt RF
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f'train: {X_train.shape[0]} days | test: {X_test.shape[0]} days')

## 4. Train Models

In [ ]:
# --- linear regression ---
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)

# --- random forest ---
rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

print('both models trained!')

## 5. Evaluate Both Models

In [ ]:
def evaluate(name, y_true, y_pred):
    """print MAE, RMSE, and R² for a model"""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f'{name}')
    print(f'  MAE:  ${mae:.2f}  |  RMSE: ${rmse:.2f}  |  R²: {r2:.4f}')
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2}

lr_metrics = evaluate('Linear Regression', y_test, lr_preds)
rf_metrics = evaluate('Random Forest    ', y_test, rf_preds)

## 6. Plot Actual vs Predicted

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

for ax, preds, name, color in zip(
    axes,
    [lr_preds, rf_preds],
    ['Linear Regression', 'Random Forest'],
    ['steelblue', 'darkorange']
):
    ax.plot(dates_test, y_test, label='actual close', color='black', linewidth=1.5)
    ax.plot(dates_test, preds, label=f'{name} prediction', color=color, linewidth=1.2, alpha=0.8)
    ax.set_title(f'{name} — Predicted vs Actual Close Price (AAPL)')
    ax.set_ylabel('Price (USD)')
    ax.legend()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))

plt.xlabel('Date')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 7. Feature Importance (Random Forest)

In [ ]:
# random forest can tell us which features it relied on most
importances = pd.Series(rf.feature_importances_, index=feature_cols)
importances_sorted = importances.sort_values(ascending=True)

plt.figure(figsize=(8, 5))
importances_sorted.plot(kind='barh', color='steelblue')
plt.title('Random Forest Feature Importances')
plt.xlabel('importance score')
plt.tight_layout()
plt.show()

## Key Findings

- **Random Forest outperforms Linear Regression** — it captures non-linear patterns in price movements that LR can't.
- The **20-day moving average (MA_20)** and **High** are the most important features — price context matters more than daily volume.
- Both models struggle during sudden market events (high prediction error spikes around volatile periods).
- Next-day price prediction is inherently noisy — even a small MAE of \$2-3 is reasonable for stock data.

> **Note:** This model is for educational purposes only — stock prices are influenced by news, sentiment, and macro events that no feature set fully captures.
